# 🤖 FAIRAgent Prototype — Agentic FAIR Assessment with LangGraph

## Architecture Decision Summary

Before writing code, here is the reasoning behind each key design choice:

### Should we use a Data Catalog, MDM, or the raw database?

| Option | Pros | Cons | FAIR Fit |
|---|---|---|---|
| **Raw Database** | Always available, no extra infra | Only schema/data, no provenance/lineage | Partial — covers I/R indicators |
| **MDM System** | Golden records, entity resolution | Focused on deduplication, not documentation | Weak — MDM ≠ FAIR metadata |
| **Data Catalog** (OpenMetadata, DataHub) | Lineage, ownership, tags, descriptions, policies | Requires catalog deployment | ✅ Best — maps directly to all FAIR indicators |
| **Database + Metadata Layer** (our prototype) | Simulates catalog via mock API, works without catalog | Mocked only | ✅ Best for prototyping |

**Decision**: The agent inspects **two sources in parallel**:
1. **Schema API** — the actual database schema (tables, columns, types, PKs, FKs)
2. **Metadata API** — a simulated data catalog document (descriptions, license, PIDs, ontology links, provenance)

This maps to reality: you'd point the agent at OpenMetadata or DataHub and the agent would call both the schema inspection endpoint and the catalog metadata endpoint.

### Why LangGraph over LlamaIndex?

- **LangGraph** — stateful `StateGraph`, first-class human-in-the-loop, conditional edges, streaming. Best for orchestrated multi-step pipelines with SME approval.
- **LlamaIndex** — excellent for RAG and document retrieval but less natural for stateful approval workflows.

### Agent Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                        FAIRAgent Graph                        │
│                                                              │
│  [START] ──► [discover] ──► [assess_F] ──► [assess_A]       │
│                                  │               │           │
│                             [assess_I] ──► [assess_R]        │
│                                                  │           │
│                              [compile_report]    │           │
│                                     │            │           │
│                              [sme_review] ◄──────┘           │
│                             (human-in-loop)                  │
│                                     │                        │
│                    ┌────────────────┴──────────────┐         │
│                    ▼                               ▼         │
│             [approved]                    [request_revision] │
│                    │                               │         │
│                  [END]                       [assess_F] ...  │
└──────────────────────────────────────────────────────────────┘
```

## Section 1 — Install & Import Required Libraries

In [16]:
%pip install -q \
    "langgraph>=1.0,<1.1" \
    "langchain-core>=1.0,<2.0" \
    "langchain-openai>=0.3" \
    fastapi "uvicorn[standard]" httpx \
    pandas plotly ipywidgets \
    "pydantic>=2.0" \
    nest_asyncio \
    python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import os, json, threading, time, textwrap
from typing import Annotated, Any
from dotenv import load_dotenv

import pandas as pd
import plotly.graph_objects as go
import httpx
import nest_asyncio
import ipywidgets as widgets
from IPython.display import display, clear_output

# Allow asyncio event loops inside Jupyter
nest_asyncio.apply()

# ── LangGraph / LangChain ──────────────────────────────────────────────────
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI

# ── FastAPI (mock database server) ────────────────────────────────────────
from fastapi import FastAPI
import uvicorn

load_dotenv()

# Set your OpenAI API key here or in a .env file as OPENAI_API_KEY
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_API_KEY_HERE")
LLM_MODEL = os.getenv("FAIR_LLM_MODEL", "gpt-5.2")

print(f"✅ Libraries loaded. LLM model: {LLM_MODEL}")
print(f"   API key configured: {'Yes' if OPENAI_API_KEY != 'YOUR_API_KEY_HERE' else '⚠️  Not set — set OPENAI_API_KEY in .env'}")

✅ Libraries loaded. LLM model: gpt-5.2
   API key configured: Yes


## Section 2 — Mock Database API Setup

We build a **FastAPI** server that simulates what you'd get from a real database + data catalog combination.  
It exposes four endpoints the FAIRAgent will call:

| Endpoint | Maps to FAIR | What it returns |
|---|---|---|
| `GET /schema` | F1, F2, I1, I2 | Table names, column names, types, PKs/FKs |
| `GET /metadata` | F1–F4, A1, R1.1–R1.3 | Dataset title, owner, licence, PIDs, ontology links |
| `GET /records?limit=5` | A1, I1, I2 | Sample data rows |
| `GET /lineage` | R1.2 | Provenance chain (source → transformation → current) |
| `GET /access` | A1.1, A1.2 | Access protocol, auth type, endpoint URL |

> **Production swap**: Replace `MOCK_API_BASE` with your OpenMetadata, DataHub, or internal catalog base URL and update `fetch_*` functions accordingly.

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# Mock dataset: a pharma in-vitro PD assay study
# This simulates what a real database + data catalog would return.
# ─────────────────────────────────────────────────────────────────────────────

MOCK_SCHEMA = {
    "dataset_id": "STUDY-INV-001",
    "tables": [
        {
            "name": "compounds",
            "columns": [
                {"name": "compound_id",  "type": "VARCHAR(50)", "primary_key": True,  "ontology": "ChEMBL:CHEMBL25"},
                {"name": "compound_name","type": "VARCHAR(200)","primary_key": False, "ontology": None},
                {"name": "smiles",       "type": "TEXT",        "primary_key": False, "ontology": "CHEMINF:000018"},
                {"name": "inchi_key",    "type": "VARCHAR(27)", "primary_key": False, "ontology": "CHEMINF:000399"},
                {"name": "target_gene",  "type": "VARCHAR(50)", "primary_key": False, "ontology": "HGNC:1097"},
            ]
        },
        {
            "name": "assay_results",
            "columns": [
                {"name": "result_id",    "type": "UUID",        "primary_key": True,  "ontology": None},
                {"name": "compound_id",  "type": "VARCHAR(50)", "primary_key": False, "foreign_key": "compounds.compound_id"},
                {"name": "assay_type",   "type": "VARCHAR(100)","primary_key": False, "ontology": "OBI:0001318"},
                {"name": "value",        "type": "FLOAT",       "primary_key": False, "ontology": None},
                {"name": "unit",         "type": "VARCHAR(20)", "primary_key": False, "ontology": "UO:0000064"},
                {"name": "cell_line",    "type": "VARCHAR(100)","primary_key": False, "ontology": "CLO:0000019"},
                {"name": "created_at",   "type": "TIMESTAMP",   "primary_key": False, "ontology": None},
                {"name": "created_by",   "type": "VARCHAR(100)","primary_key": False, "ontology": None},
            ]
        }
    ]
}

MOCK_METADATA = {
    "dataset_id": "STUDY-INV-001",
    "title": "In-vitro PD Assay Panel — Compound Series Alpha",
    "description": "IC50 measurements for 48 compounds against EGFR using cell viability assay.",
    "owner": "Dr. Jane Smith <jane.smith@example.org>",
    "created_at": "2024-03-15T09:00:00Z",
    "modified_at": "2024-11-20T14:30:00Z",
    "persistent_identifier": "https://doi.org/10.99999/study-inv-001",  # F1
    "global_identifier_system": "DOI",                                   # F1
    "catalog_url": "https://datacatalog.example.org/datasets/STUDY-INV-001",  # F4
    "license": {
        "name": "CC BY 4.0",
        "url": "https://creativecommons.org/licenses/by/4.0/",
        "spdx": "CC-BY-4.0",                                             # R1.1
        "machine_readable": True
    },
    "access": {
        "protocol": "HTTPS",                                             # A1
        "endpoint": "https://api.example.org/datasets/STUDY-INV-001",
        "authentication": "OAuth2 / API token",
        "is_open_protocol": True,                                        # A1.1
        "access_request_url": "https://dac.example.org/request"         # A1.2
    },
    "metadata_persists_without_data": True,                              # A2
    "vocabularies": ["OBI", "ChEMBL", "CLO", "UO", "HGNC", "CHEMINF"], # I2
    "community_standard": "ISA-Tab",                                     # R1.3
    "provenance": {                                                      # R1.2
        "source": "Internal ELN (Benchling) — Signed audit trail",
        "transformation": "ETL pipeline v2.3",
        "alcoa_compliant": True
    },
    "related_datasets": [                                                # I3
        {"id": "STUDY-INV-000", "relation": "IsPreviousVersionOf", "url": "https://doi.org/10.99999/study-inv-000"},
        {"id": "STUDY-CLINIC-007", "relation": "IsSupplementTo",   "url": "https://doi.org/10.99999/study-clinic-007"}
    ]
}

MOCK_RECORDS = [
    {"compound_id": "CMPD-001", "assay_type": "OBI:0001318", "value": 12.4, "unit": "nM", "cell_line": "CLO:0000019"},
    {"compound_id": "CMPD-002", "assay_type": "OBI:0001318", "value": 85.1, "unit": "nM", "cell_line": "CLO:0000019"},
    {"compound_id": "CMPD-003", "assay_type": "OBI:0001318", "value": 3.2,  "unit": "nM", "cell_line": "CLO:0000019"},
]

MOCK_LINEAGE = {
    "steps": [
        {"step": 1, "description": "Raw instrument output (CSV)", "actor": "Vi-CELL XR instrument", "timestamp": "2024-03-15"},
        {"step": 2, "description": "ELN entry with ALCOA+ fields", "actor": "Dr. Jane Smith", "timestamp": "2024-03-15"},
        {"step": 3, "description": "ETL pipeline normalisation", "actor": "Automated pipeline v2.3", "timestamp": "2024-03-16"},
        {"step": 4, "description": "Data lake load (Bronze → Silver)", "actor": "DataOps team", "timestamp": "2024-03-17"},
    ]
}

print("✅ Mock dataset defined: STUDY-INV-001")
print(f"   Tables: {[t['name'] for t in MOCK_SCHEMA['tables']]}")
print(f"   License: {MOCK_METADATA['license']['spdx']}")
print(f"   PID: {MOCK_METADATA['persistent_identifier']}")

✅ Mock dataset defined: STUDY-INV-001
   Tables: ['compounds', 'assay_results']
   License: CC-BY-4.0
   PID: https://doi.org/10.99999/study-inv-001


In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# Mock API — direct Python dispatch (no HTTP server needed in notebook)
# In production, swap _get() to call a real catalog REST endpoint.
# ─────────────────────────────────────────────────────────────────────────────

import copy

_MOCK_DISPATCH = {
    "schema":   lambda: copy.deepcopy(MOCK_SCHEMA),
    "metadata": lambda: copy.deepcopy(MOCK_METADATA),
    "records":  lambda: {"records": copy.deepcopy(MOCK_RECORDS)},
    "lineage":  lambda: copy.deepcopy(MOCK_LINEAGE),
    "access":   lambda: copy.deepcopy(MOCK_METADATA["access"]),
}

def _get(endpoint: str) -> dict:
    """Simulates an HTTP GET to the data catalog API.
    Swap this body for an httpx.get() call to use a real catalog."""
    handler = _MOCK_DISPATCH.get(endpoint)
    if handler is None:
        raise ValueError(f"Unknown endpoint: {endpoint}")
    return handler()

# Smoke-test
meta = _get("metadata")
schema = _get("schema")
print("✅ Mock API dispatcher ready (no HTTP server required)")
print(f"   /metadata → {list(meta.keys())}")
print(f"   /schema   → tables: {[t['name'] for t in schema['tables']]}")

✅ Mock API dispatcher ready (no HTTP server required)
   /metadata → ['dataset_id', 'title', 'description', 'owner', 'created_at', 'modified_at', 'persistent_identifier', 'global_identifier_system', 'catalog_url', 'license', 'access', 'metadata_persists_without_data', 'vocabularies', 'community_standard', 'provenance', 'related_datasets']
   /schema   → tables: ['compounds', 'assay_results']


## Section 3 — FAIR Indicators Definition

Each indicator is modelled as a `FAIRIndicator` dataclass with:
- `id` — RDA indicator ID  
- `principle` — F / A / I / R  
- `priority` — essential / important / useful  
- `question` — the question the agent must answer  
- `api_sources` — which mock API endpoints provide evidence  
- `evidence_keys` — the specific JSON keys to inspect

In [21]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class FAIRIndicator:
    id: str
    principle: str          # F / A / I / R
    priority: str           # essential / important / useful
    name: str
    question: str
    api_sources: List[str]  # which endpoints to call
    evidence_keys: List[str]  # JSON keys to check
    weight: int = 1         # 3=essential, 2=important, 1=useful

# A representative subset of the 41 indicators — the agent will score all of these
INDICATORS: List[FAIRIndicator] = [
    # ── Findable ──────────────────────────────────────────────────────────
    FAIRIndicator("RDA-F1-01M", "F", "essential",
        "Metadata has a persistent identifier",
        "Does the metadata record have a PID (DOI, ARK, Handle)?",
        ["metadata"], ["persistent_identifier", "global_identifier_system"], weight=3),

    FAIRIndicator("RDA-F1-01D", "F", "essential",
        "Data has a persistent identifier",
        "Does the data object itself have a globally unique PID?",
        ["metadata"], ["persistent_identifier", "dataset_id"], weight=3),

    FAIRIndicator("RDA-F2-01M", "F", "essential",
        "Rich metadata provided",
        "Does the metadata include rich descriptive attributes (assay type, conditions, organism)?",
        ["metadata", "schema"], ["title", "description", "owner", "vocabularies"], weight=3),

    FAIRIndicator("RDA-F3-01M", "F", "essential",
        "Metadata includes data identifier",
        "Does the metadata record explicitly reference the data PID?",
        ["metadata"], ["persistent_identifier", "catalog_url"], weight=3),

    FAIRIndicator("RDA-F4-01M", "F", "essential",
        "Metadata is harvestable/indexed",
        "Is the metadata registered in a searchable catalog or indexed by a discovery service?",
        ["metadata"], ["catalog_url"], weight=3),

    # ── Accessible ────────────────────────────────────────────────────────
    FAIRIndicator("RDA-A1-01M", "A", "important",
        "Metadata contains access information",
        "Does the metadata describe how to access the data, including any auth requirements?",
        ["metadata", "access"], ["access"], weight=2),

    FAIRIndicator("RDA-A1.1-01D", "A", "important",
        "Data accessible via open/free protocol",
        "Can the data be retrieved using an open, free, universally implementable protocol (HTTP/HTTPS)?",
        ["access"], ["protocol", "is_open_protocol"], weight=2),

    FAIRIndicator("RDA-A1.2-01D", "A", "useful",
        "Access protocol supports auth/authorisation",
        "If access is controlled, does the protocol support authentication and is the process documented?",
        ["access"], ["authentication", "access_request_url"], weight=1),

    FAIRIndicator("RDA-A2-01M", "A", "essential",
        "Metadata persists after data removal",
        "Is the metadata guaranteed to remain accessible even if the data is removed or embargoed?",
        ["metadata"], ["metadata_persists_without_data"], weight=3),

    # ── Interoperable ─────────────────────────────────────────────────────
    FAIRIndicator("RDA-I1-01M", "I", "important",
        "Metadata uses formal knowledge representation",
        "Is the metadata encoded in a standard format (JSON-LD, RDF, schema.org)?",
        ["metadata", "schema"], ["vocabularies", "community_standard"], weight=2),

    FAIRIndicator("RDA-I1-01D", "I", "important",
        "Data uses standardised format",
        "Is the data encoded in a community-recognised format (mzML, ISA-Tab, FASTA, SDF)?",
        ["metadata"], ["community_standard"], weight=2),

    FAIRIndicator("RDA-I2-01M", "I", "important",
        "Metadata uses FAIR vocabularies",
        "Do the vocabularies used in metadata have PIDs and are they openly accessible (OBI, ChEBI, GO)?",
        ["metadata", "schema"], ["vocabularies"], weight=2),

    FAIRIndicator("RDA-I2-01D", "I", "useful",
        "Data uses FAIR vocabularies",
        "Do controlled terms in the data files reference FAIR ontology identifiers?",
        ["records", "schema"], ["ontology"], weight=1),

    FAIRIndicator("RDA-I3-01M", "I", "important",
        "Metadata has qualified references",
        "Does the metadata include typed links to related datasets or publications?",
        ["metadata"], ["related_datasets"], weight=2),

    # ── Reusable ──────────────────────────────────────────────────────────
    FAIRIndicator("RDA-R1-01M", "R", "essential",
        "Rich metadata for reuse",
        "Does the metadata provide enough attributes for a new user to decide if the data is suitable for reuse?",
        ["metadata"], ["description", "title", "owner", "provenance"], weight=3),

    FAIRIndicator("RDA-R1.1-01M", "R", "essential",
        "Metadata includes licence",
        "Does the metadata explicitly specify a reuse licence?",
        ["metadata"], ["license"], weight=3),

    FAIRIndicator("RDA-R1.1-02M", "R", "important",
        "Metadata refers to standard licence",
        "Is the licence a recognised standard (CC BY, CC0, ODbL)?",
        ["metadata"], ["license"], weight=2),

    FAIRIndicator("RDA-R1.1-03M", "R", "important",
        "Licence is machine-readable",
        "Is the licence expressed using an SPDX identifier or machine-readable URI?",
        ["metadata"], ["license"], weight=2),

    FAIRIndicator("RDA-R1.2-01M", "R", "important",
        "Metadata includes provenance",
        "Does the metadata include provenance information using a cross-domain standard (W3C PROV, PAV)?",
        ["metadata", "lineage"], ["provenance"], weight=2),

    FAIRIndicator("RDA-R1.3-01M", "R", "essential",
        "Metadata complies with community standard",
        "Does the metadata structure comply with a recognised community standard for the domain?",
        ["metadata"], ["community_standard"], weight=3),
]

print(f"✅ Defined {len(INDICATORS)} FAIR indicators")
for p in "FAIR":
    n = sum(1 for i in INDICATORS if i.principle == p)
    print(f"   {p}: {n} indicators")

✅ Defined 20 FAIR indicators
   F: 5 indicators
   A: 4 indicators
   I: 5 indicators
   R: 6 indicators


## Section 4 — LangGraph Agent Tools for FAIR Assessment

Each tool function:
1. Calls the relevant mock API endpoint(s)  
2. Extracts the evidence fields the indicator requires  
3. Returns a structured evidence dict

The LLM will call these tools to gather evidence before scoring.

In [22]:
from langchain_core.tools import tool

# ── FAIR Tools ─────────────────────────────────────────────────────────────

@tool
def check_persistent_identifier() -> dict:
    """Check whether the dataset has a globally unique persistent identifier (PID) such as a DOI or ARK.
    Returns the PID value and the identifier system used."""
    meta = _get("metadata")
    return {
        "persistent_identifier": meta.get("persistent_identifier"),
        "identifier_system": meta.get("global_identifier_system"),
        "has_pid": bool(meta.get("persistent_identifier")),
    }

@tool
def check_rich_metadata() -> dict:
    """Check whether the metadata is rich and descriptive enough for discovery.
    Returns title, description, owner, controlled vocabulary list."""
    meta = _get("metadata")
    schema = _get("schema")
    ontology_cols = [
        c["name"] for t in schema.get("tables", [])
        for c in t["columns"] if c.get("ontology")
    ]
    return {
        "title": meta.get("title"),
        "description": meta.get("description"),
        "owner": meta.get("owner"),
        "vocabularies_used": meta.get("vocabularies", []),
        "columns_with_ontology_links": ontology_cols,
        "total_columns": sum(len(t["columns"]) for t in schema.get("tables", [])),
    }

@tool
def check_catalog_registration() -> dict:
    """Check whether the metadata is registered in a searchable catalog or indexed by a discovery service."""
    meta = _get("metadata")
    return {
        "catalog_url": meta.get("catalog_url"),
        "is_indexed": bool(meta.get("catalog_url")),
    }

@tool
def check_access_protocol() -> dict:
    """Check what protocol is used to access data and whether it is open and free (HTTP/HTTPS/FTP).
    Also checks whether authentication is required and whether access request process is documented."""
    access = _get("access")
    return {
        "protocol": access.get("protocol"),
        "is_open_protocol": access.get("is_open_protocol"),
        "authentication": access.get("authentication"),
        "access_request_url": access.get("access_request_url"),
        "endpoint": access.get("endpoint"),
    }

@tool
def check_metadata_persistence() -> dict:
    """Check whether the metadata is guaranteed to persist even if the underlying data is removed."""
    meta = _get("metadata")
    return {
        "metadata_persists_without_data": meta.get("metadata_persists_without_data"),
    }

@tool
def check_interoperability() -> dict:
    """Check whether the data and metadata use community-standard formats and FAIR-compliant vocabularies.
    Returns community standard, vocabulary list, and ontology-annotated column count."""
    meta = _get("metadata")
    schema = _get("schema")
    records = _get("records")
    ontology_values_in_records = any(
        str(v).startswith(tuple(["OBI:", "CLO:", "UO:", "CHEMINF:", "HGNC:", "ChEMBL:"]))
        for row in records.get("records", [])
        for v in row.values()
    )
    return {
        "community_standard": meta.get("community_standard"),
        "vocabularies": meta.get("vocabularies", []),
        "ontology_values_in_records": ontology_values_in_records,
        "related_datasets": meta.get("related_datasets", []),
    }

@tool
def check_license_and_reuse() -> dict:
    """Check whether the metadata includes a clear, machine-readable reuse licence."""
    meta = _get("metadata")
    lic = meta.get("license", {})
    return {
        "license_name": lic.get("name"),
        "license_url": lic.get("url"),
        "spdx_identifier": lic.get("spdx"),
        "machine_readable": lic.get("machine_readable"),
        "is_standard_license": lic.get("spdx") is not None,
    }

@tool
def check_provenance() -> dict:
    """Check whether metadata includes detailed provenance information (W3C PROV, ALCOA+, audit trail)."""
    meta = _get("metadata")
    lineage = _get("lineage")
    return {
        "provenance_source": meta.get("provenance", {}).get("source"),
        "alcoa_compliant": meta.get("provenance", {}).get("alcoa_compliant"),
        "lineage_steps": len(lineage.get("steps", [])),
        "lineage_details": lineage.get("steps", []),
    }

ALL_TOOLS = [
    check_persistent_identifier,
    check_rich_metadata,
    check_catalog_registration,
    check_access_protocol,
    check_metadata_persistence,
    check_interoperability,
    check_license_and_reuse,
    check_provenance,
]

print(f"✅ {len(ALL_TOOLS)} FAIR assessment tools defined")

✅ 8 FAIR assessment tools defined


## Section 5 — Agentic Workflow Orchestration (LangGraph StateGraph)

The graph has these nodes:

| Node | What it does |
|---|---|
| `discover` | Fetches schema + metadata, stores in state |
| `assess_principle` | LLM + tools assesses all indicators for one FAIR principle (F/A/I/R) |
| `compile_report` | Aggregates all principle scores into a scorecard |
| `sme_review` | **Human-in-the-loop** — pauses for SME approval |

The state carries: raw evidence, LLM reasoning, scores, and approval status.

In [23]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import ToolMessage
import json

# ─────────────────────────────────────────────────────────────────────────────
# Graph State
# ─────────────────────────────────────────────────────────────────────────────

class FAIRAgentState(TypedDict):
    dataset_id: str
    raw_evidence: dict          # keyed by tool name
    principle_assessments: dict # F/A/I/R → list of {indicator_id, score, reasoning}
    scorecard: list             # final list of scored indicator rows
    sme_overrides: dict         # indicator_id → {score, comment}
    status: str                 # "assessing" | "awaiting_sme" | "approved" | "revision_requested"
    messages: list              # conversation log

# ─────────────────────────────────────────────────────────────────────────────
# LLM + tool binding
# ─────────────────────────────────────────────────────────────────────────────

llm = ChatOpenAI(model=LLM_MODEL, api_key=OPENAI_API_KEY, temperature=0)
llm_with_tools = llm.bind_tools(ALL_TOOLS)

TOOL_MAP = {t.name: t for t in ALL_TOOLS}

SYSTEM_PROMPT = """You are a FAIR data maturity assessor. Your job is to evaluate a dataset against
RDA FAIR Data Maturity indicators and assign a compliance score for each indicator.

Score scale (0–3):
  3 = Fully implemented
  2 = In implementation / mostly compliant  
  1 = Under consideration / partial evidence
  0 = Not implemented / no evidence

For each indicator you assess:
1. Call the relevant tool(s) to retrieve evidence
2. Analyse the evidence against the indicator criteria
3. Return a JSON object: {"indicator_id": "...", "score": 0-3, "reasoning": "...", "evidence_summary": "..."}

Be concise and evidence-based. Do not hallucinate evidence."""


def _run_llm_for_principle(principle: str, state: FAIRAgentState) -> list:
    """Run the LLM tool-use loop to assess all indicators for a given FAIR principle."""
    principle_indicators = [i for i in INDICATORS if i.principle == principle]

    indicator_list = "\n".join(
        f"- {i.id}: {i.name} — {i.question}" for i in principle_indicators
    )
    user_msg = f"""Please assess the following FAIR indicators for principle {principle}:

{indicator_list}

Use the available tools to gather evidence, then return your assessment as a JSON array,
one object per indicator with keys: indicator_id, score (0-3), reasoning, evidence_summary."""

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_msg),
    ]

    # Agentic tool-use loop
    max_steps = 10
    for _ in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            break  # LLM is done calling tools

        # Execute each tool call
        for tc in response.tool_calls:
            tool_fn = TOOL_MAP.get(tc["name"])
            if tool_fn:
                result = tool_fn.invoke(tc["args"])
                messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    # Extract final JSON assessment from last AI message
    final_text = response.content if hasattr(response, "content") else str(response)

    # Try to extract JSON array from the response
    try:
        start = final_text.find("[")
        end = final_text.rfind("]") + 1
        if start >= 0 and end > start:
            assessments = json.loads(final_text[start:end])
        else:
            # Fallback: create a minimal entry
            assessments = [
                {"indicator_id": i.id, "score": 1, "reasoning": final_text[:200],
                 "evidence_summary": "Parsing error — check raw response"}
                for i in principle_indicators
            ]
    except json.JSONDecodeError:
        assessments = [
            {"indicator_id": i.id, "score": 1, "reasoning": "JSON parse error",
             "evidence_summary": final_text[:300]}
            for i in principle_indicators
        ]
    return assessments


print("✅ LangGraph state and LLM defined")

✅ LangGraph state and LLM defined


In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# Graph Node Definitions
# ─────────────────────────────────────────────────────────────────────────────

def discover_node(state: FAIRAgentState) -> FAIRAgentState:
    """Pre-fetch all raw evidence from the mock API."""
    print("🔍 [discover] Fetching evidence from all API endpoints...")
    evidence = {}
    for endpoint in ["schema", "metadata", "records", "lineage", "access"]:
        evidence[endpoint] = _get(endpoint)
    state["raw_evidence"] = evidence
    state["status"] = "assessing"
    print(f"   Collected evidence from: {list(evidence.keys())}")
    return state


def assess_f_node(state: FAIRAgentState) -> FAIRAgentState:
    print("🤖 [assess_F] Assessing Findability indicators...")
    results = _run_llm_for_principle("F", state)
    state["principle_assessments"]["F"] = results
    print(f"   {len(results)} F indicators assessed")
    return state


def assess_a_node(state: FAIRAgentState) -> FAIRAgentState:
    print("🤖 [assess_A] Assessing Accessibility indicators...")
    results = _run_llm_for_principle("A", state)
    state["principle_assessments"]["A"] = results
    print(f"   {len(results)} A indicators assessed")
    return state


def assess_i_node(state: FAIRAgentState) -> FAIRAgentState:
    print("🤖 [assess_I] Assessing Interoperability indicators...")
    results = _run_llm_for_principle("I", state)
    state["principle_assessments"]["I"] = results
    print(f"   {len(results)} I indicators assessed")
    return state


def assess_r_node(state: FAIRAgentState) -> FAIRAgentState:
    print("🤖 [assess_R] Assessing Reusability indicators...")
    results = _run_llm_for_principle("R", state)
    state["principle_assessments"]["R"] = results
    print(f"   {len(results)} R indicators assessed")
    return state


def compile_report_node(state: FAIRAgentState) -> FAIRAgentState:
    """Flatten all principle assessments into a unified scorecard."""
    print("📊 [compile_report] Building scorecard...")
    scorecard = []
    for principle, assessments in state["principle_assessments"].items():
        for item in assessments:
            # Find the matching indicator definition for weight
            ind = next((i for i in INDICATORS if i.id == item.get("indicator_id")), None)
            scorecard.append({
                "indicator_id":    item.get("indicator_id", "?"),
                "principle":       principle,
                "priority":        ind.priority if ind else "?",
                "weight":          ind.weight if ind else 1,
                "score":           item.get("score", 0),
                "max_score":       3,
                "reasoning":       item.get("reasoning", ""),
                "evidence_summary": item.get("evidence_summary", ""),
                "sme_approved":    False,
                "sme_comment":     "",
            })
    state["scorecard"] = scorecard
    state["status"] = "awaiting_sme"
    total = sum(r["score"] * r["weight"] for r in scorecard)
    max_total = sum(r["max_score"] * r["weight"] for r in scorecard)
    print(f"   Scorecard: {len(scorecard)} indicators | Weighted total: {total}/{max_total}")
    return state


# ─────────────────────────────────────────────────────────────────────────────
# Build and Compile the Graph
# ─────────────────────────────────────────────────────────────────────────────

builder = StateGraph(FAIRAgentState)
builder.add_node("discover",        discover_node)
builder.add_node("assess_F",        assess_f_node)
builder.add_node("assess_A",        assess_a_node)
builder.add_node("assess_I",        assess_i_node)
builder.add_node("assess_R",        assess_r_node)
builder.add_node("compile_report",  compile_report_node)

builder.add_edge(START,           "discover")
builder.add_edge("discover",      "assess_F")
builder.add_edge("assess_F",      "assess_A")
builder.add_edge("assess_A",      "assess_I")
builder.add_edge("assess_I",      "assess_R")
builder.add_edge("assess_R",      "compile_report")
builder.add_edge("compile_report", END)

memory = MemorySaver()
fair_graph = builder.compile(checkpointer=memory)

print("✅ LangGraph StateGraph compiled")
print("   Nodes:", list(fair_graph.get_graph().nodes))

✅ LangGraph StateGraph compiled
   Nodes: ['__start__', 'discover', 'assess_F', 'assess_A', 'assess_I', 'assess_R', 'compile_report', '__end__']


## Section 6 — Run the Agent

Set your `OPENAI_API_KEY` in a `.env` file or the cell below, then run the agent against the mock dataset.

In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# Run the FAIRAgent end-to-end
# ─────────────────────────────────────────────────────────────────────────────

initial_state: FAIRAgentState = {
    "dataset_id": "STUDY-INV-001",
    "raw_evidence": {},
    "principle_assessments": {},
    "scorecard": [],
    "sme_overrides": {},
    "status": "starting",
    "messages": [],
}

config = {"configurable": {"thread_id": "fair-run-001"}}

print("=" * 65)
print("  FAIRAgent — Running full FAIR maturity assessment")
print("=" * 65)

final_state = fair_graph.invoke(initial_state, config=config)

print("\n" + "=" * 65)
print(f"  Assessment complete — {len(final_state['scorecard'])} indicators scored")
print("=" * 65)

  FAIRAgent — Running full FAIR maturity assessment
🔍 [discover] Fetching evidence from all API endpoints...
   Collected evidence from: ['schema', 'metadata', 'records', 'lineage', 'access']
🤖 [assess_F] Assessing Findability indicators...
   5 F indicators assessed
🤖 [assess_A] Assessing Accessibility indicators...
   4 A indicators assessed
🤖 [assess_I] Assessing Interoperability indicators...
   5 I indicators assessed
🤖 [assess_R] Assessing Reusability indicators...
   6 R indicators assessed
📊 [compile_report] Building scorecard...
   Scorecard: 20 indicators | Weighted total: 117/141

  Assessment complete — 20 indicators scored


## Section 7 — SME Review & Approval Interface

The widget below renders each scored indicator with:
- The LLM's reasoning and evidence summary  
- A score dropdown for the SME to override  
- A free-text comment box  
- An **Approve** button per indicator  

Once all indicators are approved, `APPROVED_SCORECARD` contains the final signed-off scorecard.

In [14]:
import ipywidgets as widgets
from IPython.display import display, HTML

APPROVED_SCORECARD = []  # filled by the SME review UI

SCORE_LABELS = {0: "0 — Not implemented", 1: "1 — Under consideration",
                2: "2 — In implementation", 3: "3 — Fully implemented"}
PRINCIPLE_COLORS = {"F": "#3b82f6", "A": "#10b981", "I": "#f59e0b", "R": "#ef4444"}


def build_sme_ui(scorecard: list) -> widgets.VBox:
    rows = []
    row_widgets = []

    summary_html = widgets.HTML(
        value=f"<h3 style='font-family:sans-serif'>SME Review — {len(scorecard)} indicators to approve</h3>"
    )
    rows.append(summary_html)

    for item in scorecard:
        principle = item["principle"]
        color = PRINCIPLE_COLORS.get(principle, "#6b7280")

        label = widgets.HTML(value=f"""
        <div style='border-left: 4px solid {color}; padding: 6px 12px; margin-bottom:4px;
                    font-family:sans-serif; background:#f9fafb; border-radius:4px'>
          <b style='color:{color}'>[{principle}] {item['indicator_id']}</b>
          <br/><span style='font-size:0.85em;color:#374151'><b>Evidence:</b> {item.get('evidence_summary','—')}</span>
          <br/><span style='font-size:0.85em;color:#6b7280'><b>Reasoning:</b> {item.get('reasoning','—')}</span>
        </div>""")

        score_drop = widgets.Dropdown(
            options=[(v, k) for k, v in SCORE_LABELS.items()],
            value=item["score"],
            layout=widgets.Layout(width="260px"),
        )
        comment_box = widgets.Textarea(
            placeholder="Optional comment or justification...",
            layout=widgets.Layout(width="400px", height="56px"),
        )
        approve_btn = widgets.Button(
            description="✔ Approve",
            button_style="success",
            layout=widgets.Layout(width="110px"),
        )
        status_lbl = widgets.Label(value="⏳ Pending")

        def _on_approve(btn, iid=item["indicator_id"], sd=score_drop, cb=comment_box, sl=status_lbl):
            APPROVED_SCORECARD.append({
                "indicator_id": iid,
                "sme_score": sd.value,
                "sme_comment": cb.value,
            })
            sl.value = "✅ Approved"
            btn.disabled = True

        approve_btn.on_click(_on_approve)

        row = widgets.HBox([
            widgets.VBox([label]),
            widgets.VBox([score_drop, comment_box, widgets.HBox([approve_btn, status_lbl])]),
        ])
        rows.append(row)
        rows.append(widgets.HTML("<hr style='border:none;border-top:1px solid #e5e7eb'>"))
        row_widgets.append((score_drop, comment_box, approve_btn, status_lbl))

    return widgets.VBox(rows)


# Only render if the agent has already run and produced a scorecard
if final_state.get("scorecard"):
    sme_ui = build_sme_ui(final_state["scorecard"])
    display(sme_ui)
else:
    print("⚠️  Run Section 6 first to generate the scorecard.")

## Section 8 — FAIR Scorecard & Radar Chart Visualisation

After the agent run (and optional SME overrides), this section:
1. Prints the tabular scorecard with weighted scores  
2. Renders a **Plotly radar chart** of F / A / I / R scores (0–100% per principle)

In [26]:
import pandas as pd
import plotly.graph_objects as go

# ── Apply SME overrides (if any) ────────────────────────────────────────────

scorecard = final_state.get("scorecard", [])
override_map = {o["indicator_id"]: o for o in APPROVED_SCORECARD}

for row in scorecard:
    iid = row["indicator_id"]
    if iid in override_map:
        row["score"] = override_map[iid]["sme_score"]
        row["sme_comment"] = override_map[iid]["sme_comment"]
        row["sme_approved"] = True

# ── Tabular scorecard ───────────────────────────────────────────────────────

df = pd.DataFrame(scorecard)
if not df.empty:
    df["weighted_score"] = df["score"] * df["weight"]
    df["weighted_max"]   = df["max_score"] * df["weight"]
    df["pct"]            = (df["weighted_score"] / df["weighted_max"] * 100).round(1)

    display_cols = ["indicator_id", "principle", "priority", "weight", "score", "pct", "sme_approved", "reasoning"]
    print("\n── FAIR Scorecard ─────────────────────────────────────────────")
    print(df[display_cols].to_string(index=False))

    # ── Per-principle aggregation ─────────────────────────────────────────
    principle_scores = {}
    for p in "FAIR":
        sub = df[df["principle"] == p]
        if sub.empty:
            principle_scores[p] = 0.0
        else:
            principle_scores[p] = round(
                sub["weighted_score"].sum() / sub["weighted_max"].sum() * 100, 1
            )

    total_pct = round(df["weighted_score"].sum() / df["weighted_max"].sum() * 100, 1)
    print(f"\n  Findable:       {principle_scores['F']}%")
    print(f"  Accessible:     {principle_scores['A']}%")
    print(f"  Interoperable:  {principle_scores['I']}%")
    print(f"  Reusable:       {principle_scores['R']}%")
    print(f"  ─────────────────────────────")
    print(f"  Overall FAIR:   {total_pct}%")

    # ── Radar chart ───────────────────────────────────────────────────────
    categories = ["Findable", "Accessible", "Interoperable", "Reusable"]
    values = [principle_scores[p] for p in "FAIR"]
    values_closed = values + [values[0]]    # close the polygon

    fig = go.Figure(go.Scatterpolar(
        r=values_closed,
        theta=categories + [categories[0]],
        fill="toself",
        fillcolor="rgba(99, 102, 241, 0.25)",
        line=dict(color="#6366f1", width=2),
        marker=dict(size=8, color="#6366f1"),
        name="FAIR Score",
    ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(range=[0, 100], ticksuffix="%", showticklabels=True, gridcolor="#e5e7eb"),
            angularaxis=dict(gridcolor="#e5e7eb"),
        ),
        title=dict(
            text=f"<b>FAIRAgent Assessment — {final_state['dataset_id']}</b><br>"
                 f"<sup>Overall FAIR score: {total_pct}%  |  SME overrides: {len(override_map)}</sup>",
            font=dict(size=16),
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        showlegend=False,
        width=600, height=550,
    )

    fig.show()
    print("\n✅ FAIRAgent assessment complete!")
else:
    print("⚠️  No scorecard data — run Section 6 first.")


── FAIR Scorecard ─────────────────────────────────────────────
indicator_id principle  priority  weight  score   pct  sme_approved                                                                                                                                                                                                                                                                                                                                                                                                   reasoning
  RDA-F1-01M         F essential       3      2  66.7          True                                                                                                                                                                                                                                                                                                                           A globally unique persistent identifier is present for the metadata record (DOI).
  RDA-F


✅ FAIRAgent assessment complete!
